# 00 — Qiskit Primer

**Quantum Information Processing** | p-engel & collaborator

---

This notebook introduces the core Qiskit workflow:
1. Build a `QuantumCircuit`
2. Apply gates
3. Measure
4. Simulate with Aer
5. Visualize results

**Prerequisites:** Python, NumPy, linear algebra (state vectors, matrix multiplication).

> If you're comfortable with bra-ket notation and density matrices from your QM/quantum optics background, you already have the conceptual foundation. Qiskit is just the computational interface.

## 1. Installation Check

In [ ]:
import qiskit
import qiskit_aer
import numpy as np
import matplotlib.pyplot as plt

print(f"Qiskit version : {qiskit.__version__}")
print(f"Qiskit Aer     : {qiskit_aer.__version__}")
print("All imports OK ✓")

## 2. Your First Quantum Circuit

A `QuantumCircuit(n, m)` holds `n` qubits and `m` classical bits.

All qubits start in |0⟩.

In [ ]:
from qiskit import QuantumCircuit

# 1 qubit, 1 classical bit
qc = QuantumCircuit(1, 1)

# Apply Hadamard gate → puts qubit in superposition: (|0⟩ + |1⟩)/√2
qc.h(0)

# Measure qubit 0 into classical bit 0
qc.measure(0, 0)

# Draw the circuit
qc.draw('mpl')

## 3. Statevector — Inspect the Quantum State

Before measuring, we can look at the exact statevector. This is the simulation-only view — on real hardware you can't peek at the wavefunction directly.

In [ ]:
from qiskit.quantum_info import Statevector

# Build a circuit WITHOUT measurement to inspect the state
qc_sv = QuantumCircuit(1)
qc_sv.h(0)  # Hadamard

sv = Statevector(qc_sv)
print("Statevector:", sv)
print("\nProbabilities:", sv.probabilities_dict())

# Draw on Bloch sphere
from qiskit.visualization import plot_bloch_multivector
plot_bloch_multivector(sv)

## 4. Run on the Aer Simulator

`AerSimulator` mimics a real quantum device (with optional noise). Here we run the noiseless ideal simulation.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

simulator = AerSimulator()

# Transpile circuit for the simulator backend
compiled = transpile(qc, simulator)

# Run with 1024 shots (repeated measurements)
job = simulator.run(compiled, shots=1024)
result = job.result()
counts = result.get_counts()

print("Measurement counts:", counts)
print(f"\nP(|0⟩) = {counts.get('0', 0) / 1024:.3f}")
print(f"P(|1⟩) = {counts.get('1', 0) / 1024:.3f}")

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts, title='Hadamard gate on |0⟩ — 1024 shots')

**Expected result:** ~50% |0⟩, ~50% |1⟩ — the Hadamard created an equal superposition.

The slight deviation from exactly 50/50 is quantum (shot) noise — just like photon counting statistics.

## 5. Two-Qubit Circuit — CNOT Gate

The CNOT (controlled-NOT) flips the **target** qubit if and only if the **control** qubit is |1⟩.

It's the key gate for creating **entanglement**.

In [ ]:
# Create a 2-qubit Bell state |Φ+⟩ = (|00⟩ + |11⟩)/√2
bell = QuantumCircuit(2, 2)

bell.h(0)       # Superpose qubit 0
bell.cx(0, 1)   # CNOT: qubit 0 controls qubit 1
bell.measure([0, 1], [0, 1])

bell.draw('mpl')

In [ ]:
compiled_bell = transpile(bell, simulator)
counts_bell = simulator.run(compiled_bell, shots=2048).result().get_counts()

print("Bell state counts:", counts_bell)
plot_histogram(counts_bell, title='Bell State |Φ+⟩ — only |00⟩ and |11⟩ observed')

**Key result:** You only ever measure |00⟩ or |11⟩ — never |01⟩ or |10⟩. The qubits are **maximally entangled**.

This is directly analogous to entangled photon pairs (e.g., SPDC sources) you may have worked with conceptually.

## 6. Gate Reference

The most common single-qubit gates as Qiskit calls:

| Gate | Qiskit | Matrix | Effect |
|------|--------|--------|--------|
| Hadamard | `qc.h(q)` | `1/√2 [[1,1],[1,-1]]` | \|0⟩ → \|+⟩ superposition |
| Pauli-X | `qc.x(q)` | `[[0,1],[1,0]]` | Bit flip (NOT gate) |
| Pauli-Y | `qc.y(q)` | `[[0,-i],[i,0]]` | Bit+phase flip |
| Pauli-Z | `qc.z(q)` | `[[1,0],[0,-1]]` | Phase flip |
| S gate | `qc.s(q)` | `[[1,0],[0,i]]` | 90° phase rotation |
| T gate | `qc.t(q)` | `[[1,0],[0,e^iπ/4]]` | 45° phase rotation |
| Rx(θ) | `qc.rx(theta, q)` | rotation around X | Continuous rotation |
| Ry(θ) | `qc.ry(theta, q)` | rotation around Y | Continuous rotation |
| Rz(θ) | `qc.rz(theta, q)` | rotation around Z | Continuous rotation |

In [ ]:
# Inspect any gate's unitary matrix
from qiskit.quantum_info import Operator

qc_h = QuantumCircuit(1)
qc_h.h(0)

print("Hadamard unitary:")
print(np.round(Operator(qc_h).data, 4))

## 7. Custom Unitary — Bring Your Own Matrix

If you have a unitary from theory (e.g., a beam splitter matrix, a rotation from your photonics work), you can embed it directly:

In [ ]:
from qiskit.extensions import UnitaryGate

# Example: 50/50 beam splitter unitary (matches Hadamard here)
theta = np.pi / 2  # reflectivity parameter
beam_splitter = np.array([
    [np.cos(theta/2), -np.sin(theta/2)],
    [np.sin(theta/2),  np.cos(theta/2)]
])

qc_custom = QuantumCircuit(1)
qc_custom.unitary(beam_splitter, [0], label='BS')

sv_custom = Statevector(qc_custom)
print("Output state:", sv_custom)
qc_custom.draw('mpl')

## 8. Next Steps

Continue with the next notebooks:

- **`01_single_qubit_gates.ipynb`** — Bloch sphere geometry, arbitrary rotations, gate decompositions
- **`02_entanglement_bells.ipynb`** — All four Bell states, quantum teleportation circuit
- **`03_grover_search.ipynb`** — Oracle construction, amplitude amplification, quadratic speedup

---

### Further reading
- [IBM Quantum Learning](https://learning.quantum.ibm.com/) — free structured courses
- [Qiskit documentation](https://docs.quantum.ibm.com/) — API reference
- Nielsen & Chuang Ch. 1–4 for theoretical grounding